# Superstore Sales — ETL
**Input:** `data/raw/superstore.csv` (latin-1, 9994 rows)  
**Output:** `data/cleaned/superstore_clean.csv` (UTF-8, enriched)

## 1. Load

In [1]:
import pandas as pd
import numpy as np

RAW_PATH     = '../data/raw/superstore.csv'
CLEANED_PATH = '../data/cleaned/superstore_clean.csv'

df = pd.read_csv(RAW_PATH, parse_dates=['Order Date', 'Ship Date'], encoding='latin-1')
print(f'Loaded: {df.shape}')
df.head(3)

Loaded: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


## 2. Fix Postal Code

Postal Code was read as `int64`, which stripped leading zeros from New England ZIP codes (e.g. `01040` became `1040`). Padding back to 5 characters restores them.

In [2]:
print('Before fix — short ZIPs:')
short_zips = df[df['Postal Code'].astype(str).str.len() < 5][['City', 'State', 'Postal Code']]
print(short_zips.drop_duplicates().to_string())

df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

print('\nAfter fix — sample:')
print(df[df['State'] == 'Vermont'][['City', 'State', 'Postal Code']].drop_duplicates().head())

Before fix — short ZIPs:
               City          State  Postal Code
185       Fairfield    Connecticut         6824
197       Westfield     New Jersey         7090
267      Morristown     New Jersey         7960
298      Belleville     New Jersey         7109
306        Lakewood     New Jersey         8701
313      Hackensack     New Jersey         7601
346          Lowell  Massachusetts         1852
366      Manchester    Connecticut         6040
377        Franklin  Massachusetts         2038
395         Warwick   Rhode Island         2886
422        Lawrence  Massachusetts         1841
644      Plainfield     New Jersey         7060
849          Linden     New Jersey         7036
858   New Brunswick     New Jersey         8901
871         Concord  New Hampshire         3301
912         Norwich    Connecticut         6360
1285     Providence   Rhode Island         2908
1335     Middletown    Connecticut         6457
1457    New Bedford  Massachusetts         2740
1461       Vine

## 3. Strip Whitespace from String Columns

In [3]:
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

print(f'Stripped whitespace from {len(str_cols)} columns: {list(str_cols)}')

Stripped whitespace from 14 columns: ['Order ID', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name']


## 4. Drop Redundant Columns

In [ ]:
print(f"Unique values in 'Country': {df['Country'].unique()}")

df.drop(columns=['Row ID', 'Country'], inplace=True)
print(f'Columns after drop: {df.shape[1]} → {list(df.columns)}')

before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'\nDuplicates removed: {before - len(df)}  →  {len(df)} rows remaining')

## 5. Add Derived Columns

In [ ]:
# Time breakdowns
df['Year']        = df['Order Date'].dt.year
df['Month']       = df['Order Date'].dt.month
df['Quarter']     = df['Order Date'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
df['Month Start'] = df['Order Date'].dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m-%d')

# Fulfillment time
df['Ship Lag (days)'] = (df['Ship Date'] - df['Order Date']).dt.days

# Profitability metrics
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)
df['Is Loss']         = (df['Profit'] < 0).astype(int)

# Discount bucketing
def discount_tier(d):
    if d == 0:      return 'None'
    elif d <= 0.2:  return 'Low'
    elif d <= 0.4:  return 'Medium'
    else:           return 'High'

df['Discount Tier'] = df['Discount'].apply(discount_tier)

print('New columns added:')
new_cols = ['Year', 'Month', 'Quarter', 'Month Start', 'Ship Lag (days)', 'Profit Margin %', 'Is Loss', 'Discount Tier']
print(df[new_cols].head())

## 6. Validate

In [6]:
print(f'Shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'\nData types:')
print(df.dtypes)
print(f'\nSample Profit Margin % — min: {df["Profit Margin %"].min():.1f}  max: {df["Profit Margin %"].max():.1f}')
print(f'Discount Tier counts:')
print(df['Discount Tier'].value_counts())

Shape: (9994, 27)
Missing values: 0
Duplicate rows: 1

Data types:
Order ID                   object
Order Date         datetime64[ns]
Ship Date          datetime64[ns]
Ship Mode                  object
Customer ID                object
Customer Name              object
Segment                    object
City                       object
State                      object
Postal Code                object
Region                     object
Product ID                 object
Category                   object
Sub-Category               object
Product Name               object
Sales                     float64
Quantity                    int64
Discount                  float64
Profit                    float64
Year                        int32
Month                       int32
Quarter                    object
Year-Month                 object
Ship Lag (days)             int64
Profit Margin %           float64
Is Loss                      bool
Discount Tier              object
dtype: object



## 7. Export

In [7]:
df.to_csv(CLEANED_PATH, index=False, encoding='utf-8')
print(f'Saved to {CLEANED_PATH}')
print(f'Final shape: {df.shape}')
df.head()

Saved to ../data/cleaned/superstore_clean.csv
Final shape: (9994, 27)


,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,Postal Code,...,Discount,Profit,Year,Month,Quarter,Year-Month,Ship Lag (days),Profit Margin %,Is Loss,Discount Tier
0,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,Henderson,Kentucky,42420,...,0.00,41.9136,2016,11,Q4,2016-11,3,16.00,False,None
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,Henderson,Kentucky,42420,...,0.00,219.5820,2016,11,Q4,2016-11,3,30.00,False,None
2,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,Los Angeles,California,90036,...,0.00,6.8714,2016,6,Q2,2016-06,4,47.00,False,None
3,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,Fort Lauderdale,Florida,33311,...,0.45,-383.0310,2015,10,Q4,2015-10,7,-40.00,True,High
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,Fort Lauderdale,Florida,33311,...,0.20,2.5164,2015,10,Q4,2015-10,7,11.25,False,Low
